In [2]:
from IPython.core.display import display, HTML
display(HTML("<style>.container { width:100% !important; }</style>"))

ImportError: cannot import name 'display' from 'IPython.core.display' (c:\Users\priya\anaconda3\Lib\site-packages\IPython\core\display.py)

# Lab | Natural Language Processing
### SMS: SPAM or HAM

### Let's prepare the environment

In [3]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer

- Read Data for the Fraudulent Email Kaggle Challenge
- Reduce the training set to speead up development. 

In [6]:
## Read Data for the Fraudulent Email Kaggle Challenge
data = pd.read_csv("C:\\Users\\priya\\AIcoursework\\week4\\lab-natural-language-processing\\data\\kg_train.csv",encoding='latin-1')

# Reduce the training set to speed up development. 
# Modify for final system
data = data.head(1000)
print(data.shape)
data.fillna("",inplace=True)

(1000, 2)


### Let's divide the training and test set into two partitions

In [7]:
from sklearn.model_selection import train_test_split

data_train, data_val = train_test_split(
    data,
    test_size=0.2,
    random_state=42,
    stratify=data["label"]
)

data_train = data_train.copy()
data_val = data_val.copy()

print("Train shape:", data_train.shape)
print("Validation shape:", data_val.shape)
print("\nTrain label distribution:")
print(data_train["label"].value_counts(normalize=True))
print("\nValidation label distribution:")
print(data_val["label"].value_counts(normalize=True))

Train shape: (800, 2)
Validation shape: (200, 2)

Train label distribution:
label
0    0.5575
1    0.4425
Name: proportion, dtype: float64

Validation label distribution:
label
0    0.56
1    0.44
Name: proportion, dtype: float64


## Data Preprocessing

In [8]:
import string
from nltk.corpus import stopwords
print(string.punctuation)
print(stopwords.words("english")[100:110])
from nltk.stem.snowball import SnowballStemmer
snowball = SnowballStemmer('english')

!"#$%&'()*+,-./:;<=>?@[\]^_`{|}~
['needn', "needn't", 'no', 'nor', 'not', 'now', 'o', 'of', 'off', 'on']


## Now, we have to clean the html code removing words

- First we remove inline JavaScript/CSS
- Then we remove html comments. This has to be done before removing regular tags since comments can contain '>' characters
- Next we can remove the remaining tags

In [9]:
from bs4 import BeautifulSoup, MarkupResemblesLocatorWarning
import warnings
import re

warnings.filterwarnings("ignore", category=MarkupResemblesLocatorWarning)

def remove_html(text):
    text = str(text)

    # Remove inline JavaScript / CSS
    text = re.sub(r"<script.*?>.*?</script>", " ", text, flags=re.DOTALL | re.IGNORECASE)
    text = re.sub(r"<style.*?>.*?</style>", " ", text, flags=re.DOTALL | re.IGNORECASE)

    # Remove HTML comments
    text = re.sub(r"<!--.*?-->", " ", text, flags=re.DOTALL)

    # Remove remaining HTML tags
    text = BeautifulSoup(text, "html.parser").get_text(separator=" ")

    return text

data_train["clean_html"] = data_train["text"].apply(remove_html)
data_val["clean_html"] = data_val["text"].apply(remove_html)

data_train[["text", "clean_html"]].head()

,text,clean_html
442,Dear=2C Good day hope fine=2Cdear am writting ...,Dear=2C Good day hope fine=2Cdear am writting ...
962,FROM MR HENRY KABORETHE CHIEF AUDITOR INCHARGE...,FROM MR HENRY KABORETHE CHIEF AUDITOR INCHARGE...
971,Will do.,Will do.
190,FROM THE DESK OF DR.ADAMU ISMALERAUDITING AND...,FROM THE DESK OF DR.ADAMU ISMALERAUDITING AND...
551,"Dear Friend, My name is LOI C.ESTRADA,The wife...","Dear Friend, My name is LOI C.ESTRADA,The wife..."


- Remove all the special characters
    
- Remove numbers
    
- Remove all single characters
 
- Remove single characters from the start

- Substitute multiple spaces with single space

- Remove prefixed 'b'

- Convert to Lowercase

In [10]:
import re

def normalize_text(text):
    text = str(text)

    # Remove special characters
    text = re.sub(r"[^a-zA-Z]", " ", text)

    # Remove single characters
    text = re.sub(r"\b[a-zA-Z]\b", " ", text)

    # Remove prefixed 'b'
    text = re.sub(r"\bb\s+", " ", text)

    # Substitute multiple spaces with single space
    text = re.sub(r"\s+", " ", text)

    # Convert to lowercase and strip
    text = text.lower().strip()

    return text

data_train["clean_text"] = data_train["clean_html"].apply(normalize_text)
data_val["clean_text"] = data_val["clean_html"].apply(normalize_text)

data_train[["clean_html", "clean_text"]].head()

,clean_html,clean_text
442,Dear=2C Good day hope fine=2Cdear am writting ...,dear good day hope fine cdear am writting this...
962,FROM MR HENRY KABORETHE CHIEF AUDITOR INCHARGE...,from mr henry kaborethe chief auditor incharge...
971,Will do.,will do
190,FROM THE DESK OF DR.ADAMU ISMALERAUDITING AND...,from the desk of dr adamu ismalerauditing and ...
551,"Dear Friend, My name is LOI C.ESTRADA,The wife...",dear friend my name is loi estrada the wife of...


## Now let's work on removing stopwords
Remove the stopwords.

In [11]:
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

stop_words = set(ENGLISH_STOP_WORDS)

def remove_stopwords(text):
    words = text.split()
    filtered_words = [word for word in words if word not in stop_words]
    return " ".join(filtered_words)

data_train["no_stopwords"] = data_train["clean_text"].apply(remove_stopwords)
data_val["no_stopwords"] = data_val["clean_text"].apply(remove_stopwords)

data_train[["clean_text", "no_stopwords"]].head()

,clean_text,no_stopwords
442,dear good day hope fine cdear am writting this...,dear good day hope fine cdear writting mail re...
962,from mr henry kaborethe chief auditor incharge...,mr henry kaborethe chief auditor inchargeforei...
971,will do,
190,from the desk of dr adamu ismalerauditing and ...,desk dr adamu ismalerauditing accounting manag...
551,dear friend my name is loi estrada the wife of...,dear friend loi estrada wife mr josephestrada ...


## Tame Your Text with Lemmatization
Break sentences into words, then use lemmatization to reduce them to their base form (e.g., "running" becomes "run"). See how this creates cleaner data for analysis!

In [12]:
from nltk.stem import SnowballStemmer

snowball = SnowballStemmer("english")

def normalize_tokens(text):
    tokens = text.split()
    normalized_tokens = [snowball.stem(token) for token in tokens]
    return " ".join(normalized_tokens)

data_train["preprocessed_text"] = data_train["no_stopwords"].apply(normalize_tokens)
data_val["preprocessed_text"] = data_val["no_stopwords"].apply(normalize_tokens)

data_train[["no_stopwords", "preprocessed_text"]].head()

,no_stopwords,preprocessed_text
442,dear good day hope fine cdear writting mail re...,dear good day hope fine cdear writ mail respec...
962,mr henry kaborethe chief auditor inchargeforei...,mr henri kaboreth chief auditor inchargeforeig...
971,,
190,desk dr adamu ismalerauditing accounting manag...,desk dr adamu ismaleraudit account manag bank ...
551,dear friend loi estrada wife mr josephestrada ...,dear friend loi estrada wife mr josephestrada ...


## Bag Of Words
Let's get the 10 top words in ham and spam messages (**EXPLORATORY DATA ANALYSIS**)

In [13]:
from collections import Counter

ham_words = " ".join(data_train.loc[data_train["label"] == 0, "preprocessed_text"]).split()
spam_words = " ".join(data_train.loc[data_train["label"] == 1, "preprocessed_text"]).split()

ham_top_10 = Counter(ham_words).most_common(10)
spam_top_10 = Counter(spam_words).most_common(10)

print("Top 10 HAM words:")
print(ham_top_10)
print("\nTop 10 SPAM words:")
print(spam_top_10)

Top 10 HAM words:
[('work', 97), ('presid', 97), ('state', 95), ('mr', 85), ('obama', 82), ('percent', 80), ('time', 73), ('pm', 67), ('said', 61), ('american', 61)]

Top 10 SPAM words:
[('money', 761), ('account', 705), ('bank', 686), ('fund', 606), ('transact', 440), ('transfer', 435), ('busi', 415), ('countri', 401), ('foreign', 395), ('assist', 384)]


## Extra features

In [14]:
# We add to the original dataframe two additional indicators (money symbols and suspicious words).
money_simbol_list = "|".join(["euro","dollar","pound","€",r"\$"])
suspicious_words = "|".join(["free","cheap","sex","money","account","bank","fund","transfer","transaction","win","deposit","password"])

data_train['money_mark'] = data_train['preprocessed_text'].str.contains(money_simbol_list)*1
data_train['suspicious_words'] = data_train['preprocessed_text'].str.contains(suspicious_words)*1
data_train['text_len'] = data_train['preprocessed_text'].apply(lambda x: len(x)) 

data_val['money_mark'] = data_val['preprocessed_text'].str.contains(money_simbol_list)*1
data_val['suspicious_words'] = data_val['preprocessed_text'].str.contains(suspicious_words)*1
data_val['text_len'] = data_val['preprocessed_text'].apply(lambda x: len(x)) 

data_train.head()

,text,label,clean_html,clean_text,no_stopwords,preprocessed_text,money_mark,suspicious_words,text_len
442,Dear=2C Good day hope fine=2Cdear am writting ...,1,Dear=2C Good day hope fine=2Cdear am writting ...,dear good day hope fine cdear am writting this...,dear good day hope fine cdear writting mail re...,dear good day hope fine cdear writ mail respec...,1,1,820
962,FROM MR HENRY KABORETHE CHIEF AUDITOR INCHARGE...,1,FROM MR HENRY KABORETHE CHIEF AUDITOR INCHARGE...,from mr henry kaborethe chief auditor incharge...,mr henry kaborethe chief auditor inchargeforei...,mr henri kaboreth chief auditor inchargeforeig...,0,1,1607
971,Will do.,0,Will do.,will do,,,0,0,0
190,FROM THE DESK OF DR.ADAMU ISMALERAUDITING AND...,1,FROM THE DESK OF DR.ADAMU ISMALERAUDITING AND...,from the desk of dr adamu ismalerauditing and ...,desk dr adamu ismalerauditing accounting manag...,desk dr adamu ismaleraudit account manag bank ...,1,1,313
551,"Dear Friend, My name is LOI C.ESTRADA,The wife...",1,"Dear Friend, My name is LOI C.ESTRADA,The wife...",dear friend my name is loi estrada the wife of...,dear friend loi estrada wife mr josephestrada ...,dear friend loi estrada wife mr josephestrada ...,1,1,1240


## How would work the Bag of Words with Count Vectorizer concept?

In [15]:
from sklearn.feature_extraction.text import CountVectorizer

bow_vectorizer = CountVectorizer(max_features=5000)
X_train_bow = bow_vectorizer.fit_transform(data_train["preprocessed_text"])
X_val_bow = bow_vectorizer.transform(data_val["preprocessed_text"])

print("Bag of Words train shape:", X_train_bow.shape)
print("Bag of Words validation shape:", X_val_bow.shape)

sample_vocab = list(bow_vectorizer.vocabulary_.items())[:10]
print("\nSample vocabulary items:")
print(sample_vocab)

Bag of Words train shape: (800, 5000)
Bag of Words validation shape: (200, 5000)

Sample vocabulary items:
[('dear', np.int64(926)), ('good', np.int64(1845)), ('day', np.int64(915)), ('hope', np.int64(2051)), ('fine', np.int64(1585)), ('writ', np.int64(4850)), ('mail', np.int64(2705)), ('respect', np.int64(3609)), ('heart', np.int64(1967)), ('tear', np.int64(4218))]


## TF-IDF

- Load the vectorizer

- Vectorize all dataset

- print the shape of the vetorized dataset

In [16]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf_vectorizer = TfidfVectorizer(max_features=5000)
X_train_tfidf = tfidf_vectorizer.fit_transform(data_train["preprocessed_text"])
X_val_tfidf = tfidf_vectorizer.transform(data_val["preprocessed_text"])

print("TF-IDF train shape:", X_train_tfidf.shape)
print("TF-IDF validation shape:", X_val_tfidf.shape)

TF-IDF train shape: (800, 5000)
TF-IDF validation shape: (200, 5000)


## And the Train a Classifier?

In [17]:
from scipy.sparse import hstack, csr_matrix
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

y_train = data_train["label"]
y_val = data_val["label"]

extra_train = csr_matrix(data_train[["money_mark", "suspicious_words", "text_len"]].values)
extra_val = csr_matrix(data_val[["money_mark", "suspicious_words", "text_len"]].values)

experiments = {
    "BoW only": (X_train_bow, X_val_bow),
    "TF-IDF only": (X_train_tfidf, X_val_tfidf),
    "BoW + extra features": (hstack([X_train_bow, extra_train]), hstack([X_val_bow, extra_val])),
    "TF-IDF + extra features": (hstack([X_train_tfidf, extra_train]), hstack([X_val_tfidf, extra_val]))
}

results = []

for name, (Xtr, Xva) in experiments.items():
    model = MultinomialNB()
    model.fit(Xtr, y_train)
    preds = model.predict(Xva)
    acc = accuracy_score(y_val, preds)
    results.append((name, acc))
    print(f"{name}: {acc:.4f}")

results_df = pd.DataFrame(results, columns=["Model", "Validation Accuracy"]).sort_values(
    by="Validation Accuracy", ascending=False
)
print("\nValidation results:")
display(results_df)

best_model_name = results_df.iloc[0]["Model"]
print("Best model:", best_model_name)

best_X_train, best_X_val = experiments[best_model_name]
best_model = MultinomialNB()
best_model.fit(best_X_train, y_train)
best_preds = best_model.predict(best_X_val)

print("\nClassification report for best model:")
print(classification_report(y_val, best_preds))
print("Confusion matrix:")
print(confusion_matrix(y_val, best_preds))

BoW only: 0.9650
TF-IDF only: 0.9600
BoW + extra features: 0.9500
TF-IDF + extra features: 0.9200

Validation results:


,Model,Validation Accuracy
0,BoW only,0.965
1,TF-IDF only,0.960
2,BoW + extra features,0.950
3,TF-IDF + extra features,0.920


Best model: BoW only

Classification report for best model:
              precision    recall  f1-score   support

           0       0.99      0.95      0.97       112
           1       0.94      0.99      0.96        88

    accuracy                           0.96       200
   macro avg       0.96      0.97      0.96       200
weighted avg       0.97      0.96      0.97       200

Confusion matrix:
[[106   6]
 [  1  87]]


### Extra Task - Implement a SPAM/HAM classifier

https://www.kaggle.com/t/b384e34013d54d238490103bc3c360ce

The classifier can not be changed!!! It must be the MultinimialNB with default parameters!

Your task is to **find the most relevant features**.

For example, you can test the following options and check which of them performs better:
- Using "Bag of Words" only
- Using "TF-IDF" only
- Bag of Words + extra flags (money_mark, suspicious_words, text_len)
- TF-IDF + extra flags


You can work with teams of two persons (recommended).

In [19]:
# Load official test set
test_data = pd.read_csv("C:\\Users\\priya\\AIcoursework\\week4\\lab-natural-language-processing\\data\\kg_test.csv", encoding="latin-1")
test_data.fillna("", inplace=True)

# Apply the same preprocessing pipeline
test_data["clean_html"] = test_data["text"].apply(remove_html)
test_data["clean_text"] = test_data["clean_html"].apply(normalize_text)
test_data["no_stopwords"] = test_data["clean_text"].apply(remove_stopwords)
test_data["preprocessed_text"] = test_data["no_stopwords"].apply(normalize_tokens)

test_data["money_mark"] = test_data["preprocessed_text"].str.contains(money_simbol_list)*1
test_data["suspicious_words"] = test_data["preprocessed_text"].str.contains(suspicious_words)*1
test_data["text_len"] = test_data["preprocessed_text"].apply(lambda x: len(x))

# Retrain the best setup on the full training data
full_data = data.copy()
full_data["clean_html"] = full_data["text"].apply(remove_html)
full_data["clean_text"] = full_data["clean_html"].apply(normalize_text)
full_data["no_stopwords"] = full_data["clean_text"].apply(remove_stopwords)
full_data["preprocessed_text"] = full_data["no_stopwords"].apply(normalize_tokens)

full_data["money_mark"] = full_data["preprocessed_text"].str.contains(money_simbol_list)*1
full_data["suspicious_words"] = full_data["preprocessed_text"].str.contains(suspicious_words)*1
full_data["text_len"] = full_data["preprocessed_text"].apply(lambda x: len(x))

if best_model_name == "BoW only":
    final_vectorizer = CountVectorizer(max_features=5000)
    X_full = final_vectorizer.fit_transform(full_data["preprocessed_text"])
    X_test_final = final_vectorizer.transform(test_data["preprocessed_text"])

elif best_model_name == "TF-IDF only":
    final_vectorizer = TfidfVectorizer(max_features=5000)
    X_full = final_vectorizer.fit_transform(full_data["preprocessed_text"])
    X_test_final = final_vectorizer.transform(test_data["preprocessed_text"])

elif best_model_name == "BoW + extra features":
    final_vectorizer = CountVectorizer(max_features=5000)
    X_full_text = final_vectorizer.fit_transform(full_data["preprocessed_text"])
    X_test_text = final_vectorizer.transform(test_data["preprocessed_text"])
    X_full = hstack([X_full_text, csr_matrix(full_data[["money_mark", "suspicious_words", "text_len"]].values)])
    X_test_final = hstack([X_test_text, csr_matrix(test_data[["money_mark", "suspicious_words", "text_len"]].values)])

else:
    final_vectorizer = TfidfVectorizer(max_features=5000)
    X_full_text = final_vectorizer.fit_transform(full_data["preprocessed_text"])
    X_test_text = final_vectorizer.transform(test_data["preprocessed_text"])
    X_full = hstack([X_full_text, csr_matrix(full_data[["money_mark", "suspicious_words", "text_len"]].values)])
    X_test_final = hstack([X_test_text, csr_matrix(test_data[["money_mark", "suspicious_words", "text_len"]].values)])

final_model = MultinomialNB()
final_model.fit(X_full, full_data["label"])
test_predictions = final_model.predict(X_test_final)

submission = pd.DataFrame({
    "Id": range(len(test_predictions)),
    "Prediction": test_predictions
})

submission.to_csv("submission.csv", index=False)

print("submission.csv saved successfully")
submission.head()

submission.csv saved successfully


,Id,Prediction
0,0,1
1,1,0
2,2,0
3,3,0
4,4,1
